<a href="https://colab.research.google.com/github/Azharbalouch-456/Pakistan-House-Price-Prediction/blob/main/Pakistan_House_Price_Prediction_(Machine_Learning).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Import Libraries & Load Data

In [524]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

In [525]:
df = pd.read_csv("/content/sample_data/Entities.csv")
df.head()

,Unnamed: 0,property_id,location_id,page_url,property_type,price,location,city,province_name,latitude,longitude,baths,purpose,bedrooms,date_added,agency,agent,Total_Area
0,0,237062,3325,https://www.zameen.com/Property/g_10_g_10_2_gr...,Flat,10000000,G-10,Islamabad,Islamabad Capital,33.679890,73.012640,2,For Sale,2,2/4/2019,NaN,NaN,1089.004
1,1,346905,3236,https://www.zameen.com/Property/e_11_2_service...,Flat,6900000,E-11,Islamabad,Islamabad Capital,33.700993,72.971492,3,For Sale,3,5/4/2019,NaN,NaN,15246.056
2,2,386513,764,https://www.zameen.com/Property/islamabad_g_15...,House,16500000,G-15,Islamabad,Islamabad Capital,33.631486,72.926559,6,For Sale,5,7/17/2019,NaN,NaN,2178.008
3,3,656161,340,https://www.zameen.com/Property/islamabad_bani...,House,43500000,Bani Gala,Islamabad,Islamabad Capital,33.707573,73.151199,4,For Sale,4,4/5/2019,NaN,NaN,10890.000
4,4,841645,3226,https://www.zameen.com/Property/dha_valley_dha...,House,7000000,DHA Defence,Islamabad,Islamabad Capital,33.492591,73.301339,3,For Sale,3,7/10/2019,Easy Property,Muhammad Junaid Ceo Muhammad Shahid Director,2178.008


# 2. Data Cleaning & Outlier Removal

In [526]:
df.drop(columns=["agency", "agent"], inplace=True, errors='ignore')

irrevelent_columns = ['Unnamed: 0', 'property_id', 'location_id', 'page_url', 'date_added']
df = df.drop(columns=irrevelent_columns, errors='ignore')

df = df[df['purpose'] == 'For Sale']

bad_rows = df[(df["price"] < 500000) | (df["Total_Area"] <= 0) | (df["baths"] > 10)].index
df = df.drop(index=bad_rows)

q_low_price, q_high_price = df["price"].quantile([0.02, 0.98])
q_low_area, q_high_area = df["Total_Area"].quantile([0.02, 0.98])

df_clean = df[
    (df['price'] >= q_low_price) &
    (df['price'] <= q_high_price) &
    (df['Total_Area'] >= q_low_area) &
    (df['Total_Area'] <= q_high_area)
].copy()

# 3. Feature Engineering & Encoding

In [527]:
df_clean.drop(columns=["purpose"], inplace=True, errors='ignore')

top_values = df_clean["location"].value_counts().head(30).index
df_clean["location"] = df_clean["location"].apply(lambda x: x if x in top_values else "Others")

df_encoded = pd.get_dummies(df_clean, columns=['property_type', 'city', 'province_name', 'location'], drop_first=True)

# 4. Model Training & Evaluation

In [528]:
X_clean = df_encoded.drop(columns=['price'])
X_clean['Total_Area'] = np.log1p(X_clean['Total_Area'])
y_clean = np.log1p(df_encoded['price'])

X_tr, X_te, y_tr, y_te = train_test_split(X_clean, y_clean, test_size=0.2, random_state=42)

In [529]:
model_new = LinearRegression()
model_new.fit(X_tr, y_tr)
y_pr = model_new.predict(X_te)

print("Linear Regression R2 Score:", r2_score(y_te, y_pr))
print("Linear Regression MSE:", mean_squared_error(y_te, y_pr))

Linear Regression R2 Score: 0.5535912256564974
Linear Regression MSE: 0.31905168584085714


In [530]:
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_tr, y_tr)
y_pr_rf = rf_model.predict(X_te)

print("Random Forest R2 Score:", r2_score(y_te, y_pr_rf))
print("Random Forest MSE:", mean_squared_error(y_te, y_pr_rf))

Random Forest R2 Score: 0.9183462535811866
Random Forest MSE: 0.05835854254535318


# 5. Actual vs Predicted Comparison

In [531]:
check_new = df_clean.loc[X_te.index, ['city', 'property_type', 'bedrooms', 'baths', 'Total_Area', 'price', 'location']].copy()
check_new['LR_Predicted'] = np.expm1(y_pr).round(0)
check_new["RF-Predicted"] = np.expm1(y_pr_rf).round(0)
check_new.head(10)

,city,property_type,bedrooms,baths,Total_Area,price,location,LR_Predicted,RF-Predicted
27345,Lahore,House,5,5,1905.757,20000000,Allama Iqbal Town,18775500.0,15975259.0
140716,Lahore,House,3,4,1361.255,10000000,Paragon City,12293039.0,10484211.0
51039,Islamabad,House,5,7,17968.566,25000000,E-11,47265694.0,28735295.0
8895,Islamabad,House,5,4,1905.757,24000000,G-13,19746156.0,25879873.0
61120,Islamabad,House,6,7,2178.008,26500000,Others,19402156.0,28252274.0
30440,Karachi,House,3,2,13068.048,8000000,Scheme 33,15836159.0,12188888.0
12227,Rawalpindi,House,9,5,1633.506,8500000,Others,20219625.0,7428334.0
90319,Lahore,House,5,5,2722.510,18500000,Others,16061927.0,22366154.0
98506,Lahore,House,5,5,2722.510,22500000,Johar Town,19608555.0,28385208.0
10537,Lahore,House,3,2,816.753,6000000,Others,8841923.0,5424850.0


# 6. Save Model

In [532]:
joblib.dump(rf_model, "pakistan_house_price_model.pkl")

['pakistan_house_price_model.pkl']

In [533]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
# import warnings
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
# warnings.filterwarnings('ignore')



In [534]:
df = pd.read_csv("/content/sample_data/Entities.csv")


df
# sows the all info of data
# df.info()

# from google.colab import drive
# drive.mount('/content/drive')


,Unnamed: 0,property_id,location_id,page_url,property_type,price,location,city,province_name,latitude,longitude,baths,purpose,bedrooms,date_added,agency,agent,Total_Area
0,0,237062,3325,https://www.zameen.com/Property/g_10_g_10_2_gr...,Flat,10000000,G-10,Islamabad,Islamabad Capital,33.679890,73.012640,2,For Sale,2,2/4/2019,NaN,NaN,1089.004
1,1,346905,3236,https://www.zameen.com/Property/e_11_2_service...,Flat,6900000,E-11,Islamabad,Islamabad Capital,33.700993,72.971492,3,For Sale,3,5/4/2019,NaN,NaN,15246.056
2,2,386513,764,https://www.zameen.com/Property/islamabad_g_15...,House,16500000,G-15,Islamabad,Islamabad Capital,33.631486,72.926559,6,For Sale,5,7/17/2019,NaN,NaN,2178.008
3,3,656161,340,https://www.zameen.com/Property/islamabad_bani...,House,43500000,Bani Gala,Islamabad,Islamabad Capital,33.707573,73.151199,4,For Sale,4,4/5/2019,NaN,NaN,10890.000
4,4,841645,3226,https://www.zameen.com/Property/dha_valley_dha...,House,7000000,DHA Defence,Islamabad,Islamabad Capital,33.492591,73.301339,3,For Sale,3,7/10/2019,Easy Property,Muhammad Junaid Ceo Muhammad Shahid Director,2178.008
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
168441,168441,17355248,6754,https://www.zameen.com/Property/gulshan_e_maym...,House,26500000,Gadap Town,Karachi,Sindh,25.029909,67.137192,0,For Sale,6,7/18/2019,Al Shahab Enterprises,Shahmir,26136.096
168442,168442,17355249,680,https://www.zameen.com/Property/gadap_town_gul...,House,12500000,Gadap Town,Karachi,Sindh,25.017951,67.136393,0,For Sale,3,7/18/2019,Al Shahab Enterprises,Shahmir,2178.008
168443,168443,17355250,6757,https://www.zameen.com/Property/gulshan_e_maym...,House,27000000,Gadap Town,Karachi,Sindh,25.015384,67.116330,0,For Sale,6,7/18/2019,Al Shahab Enterprises,Shahmir,26136.096
168444,168444,17355251,6752,https://www.zameen.com/Property/gulshan_e_maym...,House,11000000,Gadap Town,Karachi,Sindh,25.013265,67.120818,0,For Sale,3,7/18/2019,Al Shahab Enterprises,Shahmir,21235.578


In [535]:
df.describe() # shows the maximum values of columns

,Unnamed: 0,property_id,location_id,price,latitude,longitude,baths,bedrooms,Total_Area
count,168446.000000,1.684460e+05,168446.000000,1.684460e+05,168446.000000,168446.000000,168446.000000,168446.000000,1.684460e+05
mean,84222.500000,1.559626e+07,4375.936395,1.776576e+07,29.859519,71.239804,2.874227,3.179422,1.394239e+04
std,48626.316059,2.251207e+06,3776.561581,3.531003e+07,3.807870,3.133042,2.463400,1.971401,8.623647e+05
min,0.000000,8.657500e+04,1.000000,0.000000e+00,11.052446,25.906027,0.000000,0.000000,0.000000e+00
25%,42111.250000,1.488320e+07,1058.000000,1.750000e+05,24.948536,67.130363,0.000000,2.000000,1.905757e+03
50%,84222.500000,1.665851e+07,3286.000000,8.500000e+06,31.459784,73.056182,3.000000,3.000000,4.356016e+03
75%,126333.750000,1.708662e+07,7220.000000,1.950000e+07,33.560887,73.259870,4.000000,4.000000,1.197904e+04
max,168445.000000,1.735772e+07,14220.000000,2.000000e+09,73.184088,80.161430,403.000000,68.000000,3.387988e+08


In [536]:
df.isnull().sum() # It shows us the sum of all data set null values if anynumber exist showing after this command in oiutput means that column has null values except zero 0

,0
Unnamed: 0,0
property_id,0
location_id,0
page_url,0
property_type,0
price,0
location,0
city,0
province_name,0
latitude,0


In [537]:
# df["agency"] = df["agency"].fillna("Unknown")
# df["agent"] = df["agent"].fillna("Unknown")
df

,Unnamed: 0,property_id,location_id,page_url,property_type,price,location,city,province_name,latitude,longitude,baths,purpose,bedrooms,date_added,agency,agent,Total_Area
0,0,237062,3325,https://www.zameen.com/Property/g_10_g_10_2_gr...,Flat,10000000,G-10,Islamabad,Islamabad Capital,33.679890,73.012640,2,For Sale,2,2/4/2019,NaN,NaN,1089.004
1,1,346905,3236,https://www.zameen.com/Property/e_11_2_service...,Flat,6900000,E-11,Islamabad,Islamabad Capital,33.700993,72.971492,3,For Sale,3,5/4/2019,NaN,NaN,15246.056
2,2,386513,764,https://www.zameen.com/Property/islamabad_g_15...,House,16500000,G-15,Islamabad,Islamabad Capital,33.631486,72.926559,6,For Sale,5,7/17/2019,NaN,NaN,2178.008
3,3,656161,340,https://www.zameen.com/Property/islamabad_bani...,House,43500000,Bani Gala,Islamabad,Islamabad Capital,33.707573,73.151199,4,For Sale,4,4/5/2019,NaN,NaN,10890.000
4,4,841645,3226,https://www.zameen.com/Property/dha_valley_dha...,House,7000000,DHA Defence,Islamabad,Islamabad Capital,33.492591,73.301339,3,For Sale,3,7/10/2019,Easy Property,Muhammad Junaid Ceo Muhammad Shahid Director,2178.008
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
168441,168441,17355248,6754,https://www.zameen.com/Property/gulshan_e_maym...,House,26500000,Gadap Town,Karachi,Sindh,25.029909,67.137192,0,For Sale,6,7/18/2019,Al Shahab Enterprises,Shahmir,26136.096
168442,168442,17355249,680,https://www.zameen.com/Property/gadap_town_gul...,House,12500000,Gadap Town,Karachi,Sindh,25.017951,67.136393,0,For Sale,3,7/18/2019,Al Shahab Enterprises,Shahmir,2178.008
168443,168443,17355250,6757,https://www.zameen.com/Property/gulshan_e_maym...,House,27000000,Gadap Town,Karachi,Sindh,25.015384,67.116330,0,For Sale,6,7/18/2019,Al Shahab Enterprises,Shahmir,26136.096
168444,168444,17355251,6752,https://www.zameen.com/Property/gulshan_e_maym...,House,11000000,Gadap Town,Karachi,Sindh,25.013265,67.120818,0,For Sale,3,7/18/2019,Al Shahab Enterprises,Shahmir,21235.578


In [538]:
df.drop(columns=["agency", "agent"], inplace=True) # drop the columns whoich as more than 50% nulll values
df


,Unnamed: 0,property_id,location_id,page_url,property_type,price,location,city,province_name,latitude,longitude,baths,purpose,bedrooms,date_added,Total_Area
0,0,237062,3325,https://www.zameen.com/Property/g_10_g_10_2_gr...,Flat,10000000,G-10,Islamabad,Islamabad Capital,33.679890,73.012640,2,For Sale,2,2/4/2019,1089.004
1,1,346905,3236,https://www.zameen.com/Property/e_11_2_service...,Flat,6900000,E-11,Islamabad,Islamabad Capital,33.700993,72.971492,3,For Sale,3,5/4/2019,15246.056
2,2,386513,764,https://www.zameen.com/Property/islamabad_g_15...,House,16500000,G-15,Islamabad,Islamabad Capital,33.631486,72.926559,6,For Sale,5,7/17/2019,2178.008
3,3,656161,340,https://www.zameen.com/Property/islamabad_bani...,House,43500000,Bani Gala,Islamabad,Islamabad Capital,33.707573,73.151199,4,For Sale,4,4/5/2019,10890.000
4,4,841645,3226,https://www.zameen.com/Property/dha_valley_dha...,House,7000000,DHA Defence,Islamabad,Islamabad Capital,33.492591,73.301339,3,For Sale,3,7/10/2019,2178.008
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
168441,168441,17355248,6754,https://www.zameen.com/Property/gulshan_e_maym...,House,26500000,Gadap Town,Karachi,Sindh,25.029909,67.137192,0,For Sale,6,7/18/2019,26136.096
168442,168442,17355249,680,https://www.zameen.com/Property/gadap_town_gul...,House,12500000,Gadap Town,Karachi,Sindh,25.017951,67.136393,0,For Sale,3,7/18/2019,2178.008
168443,168443,17355250,6757,https://www.zameen.com/Property/gulshan_e_maym...,House,27000000,Gadap Town,Karachi,Sindh,25.015384,67.116330,0,For Sale,6,7/18/2019,26136.096
168444,168444,17355251,6752,https://www.zameen.com/Property/gulshan_e_maym...,House,11000000,Gadap Town,Karachi,Sindh,25.013265,67.120818,0,For Sale,3,7/18/2019,21235.578


In [539]:
df.isnull().sum()

,0
Unnamed: 0,0
property_id,0
location_id,0
page_url,0
property_type,0
price,0
location,0
city,0
province_name,0
latitude,0


In [540]:
df.drop(columns = ["Unnamed: 0", "property_id", "page_url", "location_id", "date_added"], inplace=True)
df

,property_type,price,location,city,province_name,latitude,longitude,baths,purpose,bedrooms,Total_Area
0,Flat,10000000,G-10,Islamabad,Islamabad Capital,33.679890,73.012640,2,For Sale,2,1089.004
1,Flat,6900000,E-11,Islamabad,Islamabad Capital,33.700993,72.971492,3,For Sale,3,15246.056
2,House,16500000,G-15,Islamabad,Islamabad Capital,33.631486,72.926559,6,For Sale,5,2178.008
3,House,43500000,Bani Gala,Islamabad,Islamabad Capital,33.707573,73.151199,4,For Sale,4,10890.000
4,House,7000000,DHA Defence,Islamabad,Islamabad Capital,33.492591,73.301339,3,For Sale,3,2178.008
...,...,...,...,...,...,...,...,...,...,...,...
168441,House,26500000,Gadap Town,Karachi,Sindh,25.029909,67.137192,0,For Sale,6,26136.096
168442,House,12500000,Gadap Town,Karachi,Sindh,25.017951,67.136393,0,For Sale,3,2178.008
168443,House,27000000,Gadap Town,Karachi,Sindh,25.015384,67.116330,0,For Sale,6,26136.096
168444,House,11000000,Gadap Town,Karachi,Sindh,25.013265,67.120818,0,For Sale,3,21235.578


In [541]:

irrevelent_columns = ['Unnamed: 0', 'property_id', 'location_id', 'page_url', 'date_added']
df = df.drop(columns=irrevelent_columns, errors='ignore')
df = df[df['purpose'] == 'For Sale']

bad_rows = df[(df["price"] < 500000) | (df["Total_Area"] <=0) | (df["baths"] > 10)].index
df = df.drop(index=bad_rows)

print(df.shape)
print(df.columns)
# print(df.shape[["Price", "Total_Area", "baths"]].describe())
df[["price", "Total_Area", "baths"]].describe()

(120517, 11)
Index(['property_type', 'price', 'location', 'city', 'province_name',
       'latitude', 'longitude', 'baths', 'purpose', 'bedrooms', 'Total_Area'],
      dtype='object')


,price,Total_Area,baths
count,1.205170e+05,1.205170e+05,120517.000000
mean,2.477238e+07,1.432157e+04,3.081781
std,3.951775e+07,1.019004e+06,2.285339
min,5.000000e+05,2.722510e+02,0.000000
25%,7.500000e+06,1.633506e+03,1.000000
50%,1.350000e+07,3.811514e+03,3.000000
75%,2.600000e+07,1.143454e+04,5.000000
max,2.000000e+09,3.387988e+08,10.000000


In [542]:
# df.drop(columns = ["purpose", "location"],  inplace = True)
df.drop(columns = ["purpose"],  inplace = True)
top_values = df["location"].value_counts().head(30).index
df["location"] = df["location"].apply(lambda x: x if x in top_values else "Others")

In [543]:
# Convert the text in numbers by get.dummies
df_incoded = pd.get_dummies(df,columns=['property_type', 'city', 'province_name', 'location'], drop_first=True)
print(df_incoded.shape)
# print(df_incoded.columns)
df_incoded.head(2)

(120517, 48)


,price,latitude,longitude,baths,bedrooms,Total_Area,property_type_Flat,property_type_House,property_type_Lower Portion,property_type_Penthouse,property_type_Room,property_type_Upper Portion,city_Islamabad,city_Karachi,city_Lahore,city_Rawalpindi,province_name_Punjab,province_name_Sindh,location_Al Rehman Garden,location_Allama Iqbal Town,location_Askari,location_Bahria Town,location_Bahria Town Karachi,location_Bahria Town Rawalpindi,location_Cantt,location_Clifton,location_DHA Defence,location_E-11,location_Federal B Area,location_G-11,location_G-13,location_Gadap Town,location_Gulberg,location_Gulistan-e-Jauhar,location_Gulshan-e-Iqbal Town,location_Jamshed Town,location_Johar Town,location_Korangi,location_Malir,location_Model Town,location_Nazimabad,location_North Karachi,location_North Nazimabad,location_Others,location_Samanabad,location_Scheme 33,location_State Life Housing Society,location_Wapda Town
0,10000000,33.679890,73.012640,2,2,1089.004,True,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False
1,6900000,33.700993,72.971492,3,3,15246.056,True,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False


In [544]:
X = df_incoded.drop(columns=["price"])
y = df_incoded["price"]
df

,property_type,price,location,city,province_name,latitude,longitude,baths,bedrooms,Total_Area
0,Flat,10000000,Others,Islamabad,Islamabad Capital,33.679890,73.012640,2,2,1089.004
1,Flat,6900000,E-11,Islamabad,Islamabad Capital,33.700993,72.971492,3,3,15246.056
2,House,16500000,Others,Islamabad,Islamabad Capital,33.631486,72.926559,6,5,2178.008
3,House,43500000,Others,Islamabad,Islamabad Capital,33.707573,73.151199,4,4,10890.000
4,House,7000000,DHA Defence,Islamabad,Islamabad Capital,33.492591,73.301339,3,3,2178.008
...,...,...,...,...,...,...,...,...,...,...
168441,House,26500000,Gadap Town,Karachi,Sindh,25.029909,67.137192,0,6,26136.096
168442,House,12500000,Gadap Town,Karachi,Sindh,25.017951,67.136393,0,3,2178.008
168443,House,27000000,Gadap Town,Karachi,Sindh,25.015384,67.116330,0,6,26136.096
168444,House,11000000,Gadap Town,Karachi,Sindh,25.013265,67.120818,0,3,21235.578


In [545]:
# Train the Model

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)
model = LinearRegression()
df

,property_type,price,location,city,province_name,latitude,longitude,baths,bedrooms,Total_Area
0,Flat,10000000,Others,Islamabad,Islamabad Capital,33.679890,73.012640,2,2,1089.004
1,Flat,6900000,E-11,Islamabad,Islamabad Capital,33.700993,72.971492,3,3,15246.056
2,House,16500000,Others,Islamabad,Islamabad Capital,33.631486,72.926559,6,5,2178.008
3,House,43500000,Others,Islamabad,Islamabad Capital,33.707573,73.151199,4,4,10890.000
4,House,7000000,DHA Defence,Islamabad,Islamabad Capital,33.492591,73.301339,3,3,2178.008
...,...,...,...,...,...,...,...,...,...,...
168441,House,26500000,Gadap Town,Karachi,Sindh,25.029909,67.137192,0,6,26136.096
168442,House,12500000,Gadap Town,Karachi,Sindh,25.017951,67.136393,0,3,2178.008
168443,House,27000000,Gadap Town,Karachi,Sindh,25.015384,67.116330,0,6,26136.096
168444,House,11000000,Gadap Town,Karachi,Sindh,25.013265,67.120818,0,3,21235.578


In [546]:
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_pred

array([16862613.49775743, 33712445.68538648, 23080005.07134664, ...,
       21220561.78464329, 29391313.97820735, 19498407.93311971],
      shape=(24104,))

In [547]:
print("R2 Score: ", r2_score(y_test, y_pred))
print("MSE: ", mean_squared_error(y_test, y_pred))



R2 Score:  0.30943090924640726
MSE:  915869302064016.9


In [548]:
result = pd.DataFrame({
    'Actual': y_test.values,
    'Predicted': y_pred
})

print(result.head(10))


     Actual     Predicted
0  16000000  1.686261e+07
1  34000000  3.371245e+07
2  18000000  2.308001e+07
3   8400000  2.601810e+07
4  60000000  4.876700e+06
5  64000000  1.899989e+06
6   4000000  1.056568e+07
7   5800000  1.258461e+07
8  18500000  2.147953e+07
9  14500000  2.523018e+07


In [549]:
print("Hamara Actual:", y_test.iloc[0])
print("Model ka Array:", y_pred[0])


Hamara Actual: 16000000
Model ka Array: 16862613.497757435


In [550]:
# Price ke sabse saste 2% aur sabse mehenge 2% gharon ki limit nikali ja rahi hai.
# or Area ke sabse chote 2% aur sabse bade 2% gharon ki limit nikali ja rahi hai.
q_low_price, q_high_price = df["price"].quantile([0.02, 0.98])
q_low_area, q_high_area =df["Total_Area"].quantile([0.02, 0.98])


In [551]:
# Clean outliers using the correct column names and matching parentheses
df_clean = df_incoded[
    (df_incoded['price'] >= q_low_price) &
    (df_incoded['price'] <= q_high_price) &
    (df_incoded['Total_Area'] >= q_low_area) &
    (df_incoded['Total_Area'] <= q_high_area)
]

print("Cleaned Data Shape:", df_clean.shape)

Cleaned Data Shape: (112684, 48)


In [552]:
X_clean = df_incoded.drop(columns= ["price"])
y_clean = np.log1p(df_incoded["price"])
df_incoded

,price,latitude,longitude,baths,bedrooms,Total_Area,property_type_Flat,property_type_House,property_type_Lower Portion,property_type_Penthouse,property_type_Room,property_type_Upper Portion,city_Islamabad,city_Karachi,city_Lahore,city_Rawalpindi,province_name_Punjab,province_name_Sindh,location_Al Rehman Garden,location_Allama Iqbal Town,location_Askari,location_Bahria Town,location_Bahria Town Karachi,location_Bahria Town Rawalpindi,location_Cantt,location_Clifton,location_DHA Defence,location_E-11,location_Federal B Area,location_G-11,location_G-13,location_Gadap Town,location_Gulberg,location_Gulistan-e-Jauhar,location_Gulshan-e-Iqbal Town,location_Jamshed Town,location_Johar Town,location_Korangi,location_Malir,location_Model Town,location_Nazimabad,location_North Karachi,location_North Nazimabad,location_Others,location_Samanabad,location_Scheme 33,location_State Life Housing Society,location_Wapda Town
0,10000000,33.679890,73.012640,2,2,1089.004,True,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False
1,6900000,33.700993,72.971492,3,3,15246.056,True,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
2,16500000,33.631486,72.926559,6,5,2178.008,False,True,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False
3,43500000,33.707573,73.151199,4,4,10890.000,False,True,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False
4,7000000,33.492591,73.301339,3,3,2178.008,False,True,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
168441,26500000,25.029909,67.137192,0,6,26136.096,False,True,False,False,False,False,False,True,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
168442,12500000,25.017951,67.136393,0,3,2178.008,False,True,False,False,False,False,False,True,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
168443,27000000,25.015384,67.116330,0,6,26136.096,False,True,False,False,False,False,False,True,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
168444,11000000,25.013265,67.120818,0,3,21235.578,False,True,False,False,False,False,False,True,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False


In [553]:
# X_train, X_test, y_train, y_test = train_test_split(X_clean, y_clean, test_size=0.2, random=42 )
# model_new = LinearRegression()
# model_new.fit(X_train, y_train)
# y_pred = model_new.predict(X_test)
X_clean = df_clean.drop(columns=['price'])
X_clean['Total_Area'] = np.log1p(X_clean['Total_Area']) # new updatesd for check score
y_clean = np.log1p(df_clean['price'])

X_tr, X_te, y_tr, y_te = train_test_split(X_clean, y_clean, test_size=0.2, random_state=42)

model_new = LinearRegression()
model_new.fit(X_tr, y_tr)

y_pr = model_new.predict(X_te)

In [554]:
# check_new = df.loc[X_tr.index, ['city', 'property_type', 'bedrooms', 'baths', 'Total_Area', 'price']].copy()
# check_new['Model_Predicted'] = y_pr.round(0)
check_new = df.loc[X_te.index, ['city', 'property_type', 'bedrooms', 'baths', 'Total_Area', 'price', 'location']].copy()
check_new['LR_Predicted'] = np.expm1(y_pr).round(0)

# Fixed typo: changed np.exm1 to np.expm1
check_new["RF-Predicted"] = np.expm1(y_pr_rf).round(0)
# check_new['New_Model_Predicted'] = np.expm1(y_pr).round(0)

# print(check_new.head(5))
check_new.head(5)

,city,property_type,bedrooms,baths,Total_Area,price,location,LR_Predicted,RF-Predicted
27345,Lahore,House,5,5,1905.757,20000000,Allama Iqbal Town,18754008.0,15975259.0
140716,Lahore,House,3,4,1361.255,10000000,Others,10460354.0,10484211.0
51039,Islamabad,House,5,7,17968.566,25000000,E-11,47581824.0,28735295.0
8895,Islamabad,House,5,4,1905.757,24000000,G-13,19713792.0,25879873.0
61120,Islamabad,House,6,7,2178.008,26500000,Others,19127198.0,28252274.0


In [555]:
print("Accuracy check by R2", r2_score(y_te, y_pr))
print("Accuracy check by MEA", mean_squared_error(y_te, y_pr))

Accuracy check by R2 0.5570718897801898
Accuracy check by MEA 0.3165640291899704


In [556]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_tr, y_tr)

y_pr_rf = rf_model.predict(X_te)

check_new["RF-Predicted"] = np.expm1(y_pr_rf).round(0)

print("Random Forest R2 Score:", r2_score(y_te, y_pr_rf))
print("Random Forest Error:", mean_squared_error(y_te, y_pr_rf))


Random Forest R2 Score: 0.9181389821594479
Random Forest Error: 0.058506680978854246


In [557]:
check_new.head(5)


,city,property_type,bedrooms,baths,Total_Area,price,location,LR_Predicted,RF-Predicted
27345,Lahore,House,5,5,1905.757,20000000,Allama Iqbal Town,18754008.0,15967067.0
140716,Lahore,House,3,4,1361.255,10000000,Others,10460354.0,10484211.0
51039,Islamabad,House,5,7,17968.566,25000000,E-11,47581824.0,28540594.0
8895,Islamabad,House,5,4,1905.757,24000000,G-13,19713792.0,25900561.0
61120,Islamabad,House,6,7,2178.008,26500000,Others,19127198.0,28092495.0


In [558]:
import joblib

joblib.dump(rf_model, "pakistan_house_price_model.pkl")


['pakistan_house_price_model.pkl']